# Stop words des reviews Steam

Liste générique (`stopwordsiso`) + mots vides propres au corpus, détectés par fréquence documentaire. Lecture de Postgres via DuckDB.

In [ ]:
import os
from pathlib import Path

import duckdb
import stopwordsiso
from sklearn.feature_extraction.text import CountVectorizer

STEAM_LANGUAGE = "english"  # code langue Steam
ISO_LANGUAGE = "en"
SAMPLE_SIZE = 20_000
MIN_DF = 5
MAX_DF = 0.3  # au-delà, un mot présent dans 30 % des reviews devient candidat
KEEP = {"not", "no", "never", "good", "great", "bad"}  # porteurs de sentiment, à ne jamais retirer

# Charge le .env du repo sans ajouter de dépendance.
env_file = Path.cwd().parents[1] / ".env"
for line in env_file.read_text().splitlines():
    if "=" in line and not line.lstrip().startswith("#"):
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

In [ ]:
conn = duckdb.connect()
conn.execute("INSTALL postgres; LOAD postgres")
dsn = (
    f"host={os.environ['POSTGRES_HOST']} port={os.environ.get('POSTGRES_PORT', '5432')} "
    "dbname=steam_reviews_sample "  # base remplie par tools/local_sample/extract.sh
    f"user={os.environ['POSTGRES_USER']} "
    f"password={os.environ['POSTGRES_PASSWORD']}"
)
conn.execute(f"ATTACH '{dsn}' AS pg (TYPE postgres, READ_ONLY)")

In [ ]:
# postgres_query exécute le SQL côté Postgres : le LIMIT y est appliqué au lieu de rapatrier toute la vue.
sample_sql = f"""
    SELECT review_text
    FROM staging.steam_review
    WHERE language = '{STEAM_LANGUAGE}' AND review_text_length > 20
    LIMIT {SAMPLE_SIZE}
"""
# postgres_query n'accepte qu'une chaîne littérale : on double les apostrophes.
quoted = sample_sql.replace("'", "''")
reviews = [text for (text,) in conn.execute(f"SELECT review_text FROM postgres_query('pg', '{quoted}')").fetchall()]
len(reviews)

In [ ]:
generic = stopwordsiso.stopwords(ISO_LANGUAGE)
# La liste anglaise contient aussi « good » et « great » : à vérifier pour chaque langue.
sorted(generic & KEEP)

In [ ]:
# binary=True : on compte les reviews qui contiennent le mot, pas ses répétitions.
vectorizer = CountVectorizer(binary=True, min_df=MIN_DF, token_pattern=r"(?u)\b\w[\w']+\b")
matrix = vectorizer.fit_transform(reviews)

doc_freq = dict(zip(vectorizer.get_feature_names_out(), matrix.sum(axis=0).A1 / len(reviews)))
top = sorted(doc_freq.items(), key=lambda kv: kv[1], reverse=True)

for word, freq in top[:50]:
    print(f"{word:<15} {freq:6.1%}  {'générique' if word in generic else ''}")

In [ ]:
# Candidats propres au corpus : fréquents mais absents de la liste générique. À relire à la main.
corpus_candidates = [word for word, freq in top if freq > MAX_DF and word not in generic]
corpus_candidates

In [ ]:
stop_words = (generic | set(corpus_candidates)) - KEEP
len(stop_words)